In [13]:
# 1. 필수 패키지 설치 (한 번만 실행)
!pip install ultralytics norfair opencv-python-headless --quiet

# 2. 모듈 불러오기
import cv2
import numpy as np
from ultralytics import YOLO
from norfair import Detection, Tracker, draw_points
from scipy.spatial.distance import euclidean
import os

# 3. 모델 로드 및 트래커 설정
model = YOLO("yolov8n.pt")
tracker = Tracker(
    distance_function=lambda d1, d2: euclidean(d1.points[0], d2.estimate[0]),
    distance_threshold=30
)

# 4. 업로드된 영상 로드
video_path = "videoplayback.mp4"
cap = cv2.VideoCapture(video_path)
FPS = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# 5. 자르고자 하는 시간 구간 (20초 ~ 40초)
start_sec, end_sec = 20, 40
start_frame = int(start_sec * FPS)
end_frame = int(end_sec * FPS)

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

# 6. 저장할 비디오 설정
ret, sample_frame = cap.read()
if not ret:
    print("❌ 첫 프레임을 읽지 못했습니다.")
    cap.release()
    exit()

height, width = sample_frame.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("result_output.mp4", fourcc, FPS, (width, height))

# 7. 충돌 예측 파라미터
positions, speeds = {}, {}
COLLISION_THRESHOLD = 15  # 픽셀 기준 충돌 거리
SPEED_THRESHOLD = 1.0     # 상대 속도 기준 (픽셀/frame)

# 8. 프레임 분석 루프
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
for frame_num in range(start_frame, end_frame):
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]
    detections = []

    for result in results.boxes:
        cls_id = int(result.cls[0])
        conf = float(result.conf[0])
        if cls_id == 2 and conf > 0.4:  # 차량 클래스
            x1, y1, x2, y2 = map(int, result.xyxy[0])
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            detections.append(Detection(points=np.array([[cx, cy]]), scores=np.array([conf])))

    tracked_objects = tracker.update(detections=detections)

    curr_positions = {}
    for obj in tracked_objects:
        obj_id = obj.id
        cx, cy = obj.estimate[0]
        curr_positions[obj_id] = (cx, cy)

        # 속도 계산
        if obj_id in positions:
            px, py = positions[obj_id]
            vx, vy = (cx - px), (cy - py)
            speed = np.hypot(vx, vy)
            speeds[obj_id] = speed

    # 충돌 위험 판단
    danger_ids = set()
    ids = list(curr_positions.keys())
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            id1, id2 = ids[i], ids[j]
            pos1, pos2 = curr_positions[id1], curr_positions[id2]
            dist = euclidean(pos1, pos2)
            rel_speed = abs(speeds.get(id1, 0) - speeds.get(id2, 0))
            if dist < COLLISION_THRESHOLD and rel_speed > SPEED_THRESHOLD:
                danger_ids.add(id1)
                danger_ids.add(id2)

    # 시각화
    draw_points(frame, tracked_objects)

    for obj in tracked_objects:
        obj_id = obj.id
        cx, cy = obj.estimate[0]
        color = (0, 0, 255) if obj_id in danger_ids else (0, 255, 0)
        cv2.circle(frame, (int(cx), int(cy)), 5, color, -1)
        cv2.putText(frame, f'ID:{obj_id}', (int(cx), int(cy - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    out.write(frame)

    positions = curr_positions.copy()

# 9. 마무리
cap.release()
out.release()
cv2.destroyAllWindows()
print("✅ 분석 완료. 'result_output.mp4'로 저장됨.")


0: 384x640 7 cars, 2 traffic lights, 152.5ms
Speed: 3.8ms preprocess, 152.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 traffic light, 153.1ms
Speed: 3.6ms preprocess, 153.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 2 traffic lights, 152.4ms
Speed: 3.8ms preprocess, 152.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 1 traffic light, 145.7ms
Speed: 3.5ms preprocess, 145.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 bus, 1 traffic light, 147.2ms
Speed: 3.8ms preprocess, 147.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 1 traffic light, 177.8ms
Speed: 4.5ms preprocess, 177.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 1 traffic light, 155.1ms
Speed: 3.5ms preprocess, 155.1ms inference, 1.3ms postprocess per image at shape (1, 3, 

In [14]:
from google.colab import files
files.download("result_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>